# Hybrid Retriever — Semantic + BM25 (weighted RRF)

Read-only consumer of the existing index. Adds keyword (BM25) retrieval alongside the
MedEmbed cosine retrieval and fuses them with **weighted Reciprocal Rank Fusion (RRF)**.

**Why RRF instead of raw weighted scores**  
BM25 scores are unbounded (0 to ~20+); cosine scores are 0–1. Adding them directly means
trusting that "18.4" and "0.82" are comparable — they are not, and that mismatch is where
silent bad rankings hide. RRF combines by **rank**, so scale never matters:

```
score(chunk) = W_SEMANTIC · 1/(K_RRF + cosine_rank)
             + W_LEXICAL  · 1/(K_RRF + bm25_rank)
```

**Weights (recommended default for a cited medical guideline RAG):**
- `W_SEMANTIC = 0.7` — meaning-matching leads (users ask paraphrased questions)
- `W_LEXICAL  = 0.3` — keyword-matching assists (catches exact thresholds like `ACR ≥30 mg/g`)

These are a starting point. On Day 2 we measure Precision@K on a labeled eval set and adjust only if the numbers say so.

**Inputs** (produced by earlier notebooks — nothing here is modified):
- `corpus/chunks/all_chunks.jsonl` (742 chunks)
- `corpus/chunks/chroma_db/` (ChromaDB collection `ckd_guidelines`, MedEmbed 1024-dim, cosine)

**Outputs:** none written to disk — this notebook is a retrieval sandbox. The `hybrid_search()`
function is what Day-3 generation will import.

In [1]:
import json
import os
import re
import warnings
warnings.filterwarnings("ignore")

import chromadb
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

CHUNKS_PATH = "corpus/chunks/all_chunks.jsonl"
CHROMA_DIR = "corpus/chunks/chroma_db"
COLLECTION_NAME = "ckd_guidelines"
EMBED_MODEL_NAME = "abhinand/MedEmbed-large-v0.1"
QUERY_INSTRUCTION = "Represent this medical question for retrieving relevant clinical guideline passages: "

# Fusion config
W_SEMANTIC = 0.7      # weight on cosine (meaning)
W_LEXICAL  = 0.3      # weight on BM25 (keywords)
K_RRF      = 60       # RRF damping constant (standard value from the RRF paper)
CANDIDATE_POOL = 50   # how many candidates each retriever contributes before fusion

print(f"Fusion: {W_SEMANTIC:.0%} semantic / {W_LEXICAL:.0%} lexical  |  RRF k={K_RRF}  |  pool={CANDIDATE_POOL}")

Fusion: 70% semantic / 30% lexical  |  RRF k=60  |  pool=50


## Step 1 — Load chunks, Chroma collection, and MedEmbed

In [2]:
# Chunks (same order they were embedded in — index i in this list == BM25 doc i)
chunks = []
with open(CHUNKS_PATH, encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))
chunk_by_id = {c["chunk_id"]: c for c in chunks}
print(f"Loaded {len(chunks)} chunks")

# Chroma (cosine / MedEmbed)
client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_collection(COLLECTION_NAME)
assert collection.count() == len(chunks), (
    f"Chroma has {collection.count()} vectors but {len(chunks)} chunks on disk — re-run embedder.ipynb"
)
print(f"Chroma collection '{COLLECTION_NAME}': {collection.count()} vectors")
print(f"  embed_model in index: {collection.metadata.get('embed_model')}")

# MedEmbed (for encoding queries; must match the model that built the index)
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print(f"Query encoder loaded: {EMBED_MODEL_NAME} (dim {embed_model.get_sentence_embedding_dimension()})")

Loaded 742 chunks


Chroma collection 'ckd_guidelines': 742 vectors
  embed_model in index: abhinand/MedEmbed-large-v0.1


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Query encoder loaded: abhinand/MedEmbed-large-v0.1 (dim 1024)


## Step 2 — Build the BM25 index

**Medical-safe tokenization** (this matters more than the algorithm):
- lowercase (`SGLT2` == `sglt2`)
- **keep numbers** — `30`, `20`, `140` are the thresholds users search for
- keep the `≥` / `≤` math symbols our parser normalized
- **no stemming** — would corrupt drug names (`dapagliflozin` → `dapagliflozi`)
- **minimal stopwords only** — standard lists drop `not`/`no`/`without`, which is a
  clinical-meaning bug. We remove only truly empty function words.

In [3]:
# Minimal stopword list — deliberately excludes negations and clinical qualifiers.
MINIMAL_STOPWORDS = {
    "the", "a", "an", "of", "to", "in", "is", "are", "and", "or", "for",
    "on", "with", "as", "at", "by", "be", "this", "that", "these", "those",
}

# Token = run of letters/digits, OR a single >=/<= math symbol.
TOKEN_RE = re.compile(r"[a-z0-9]+|[≥≤]")


def tokenize(text: str) -> list[str]:
    toks = TOKEN_RE.findall(text.lower())
    return [t for t in toks if t not in MINIMAL_STOPWORDS]


# Build BM25 over chunk texts, in the SAME order as `chunks`
tokenized_corpus = [tokenize(c["text"]) for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)

# Sanity: show tokenization keeps numbers + symbols
sample = "Start an SGLT2i when eGFR ≥20 ml/min; ACR ≥30 mg/g is not normal."
print(f"BM25 built over {len(tokenized_corpus)} chunks")
print(f"Sample tokenization:\n  {sample}\n  -> {tokenize(sample)}")

BM25 built over 742 chunks
Sample tokenization:
  Start an SGLT2i when eGFR ≥20 ml/min; ACR ≥30 mg/g is not normal.
  -> ['start', 'sglt2i', 'when', 'egfr', '≥', '20', 'ml', 'min', 'acr', '≥', '30', 'mg', 'g', 'not', 'normal']


## Step 3 — The three retrievers

Each returns a **ranked list of chunk_ids** (best first). The individual scores are kept
for display, but fusion uses ranks only.

In [4]:
import numpy as np


def cosine_search(query: str, k: int = CANDIDATE_POOL):
    """Semantic retrieval via Chroma. Returns [(chunk_id, similarity), ...] best-first."""
    q_emb = embed_model.encode(
        [QUERY_INSTRUCTION + query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    res = collection.query(query_embeddings=q_emb.tolist(), n_results=k)
    out = []
    for cid, dist in zip(res["ids"][0], res["distances"][0]):
        out.append((cid, 1.0 - dist))    # cosine similarity
    return out


def bm25_search(query: str, k: int = CANDIDATE_POOL):
    """Lexical retrieval via BM25. Returns [(chunk_id, bm25_score), ...] best-first."""
    scores = bm25.get_scores(tokenize(query))
    top_idx = np.argsort(scores)[::-1][:k]
    return [(chunks[i]["chunk_id"], float(scores[i])) for i in top_idx]


def weighted_rrf(query: str, k: int = 5,
                 w_semantic: float = W_SEMANTIC,
                 w_lexical: float = W_LEXICAL,
                 k_rrf: int = K_RRF,
                 pool: int = CANDIDATE_POOL):
    """Weighted Reciprocal Rank Fusion of cosine + BM25.

    Each retriever contributes `pool` candidates. A chunk's fused score is the
    weighted sum of 1/(k_rrf + rank) across the lists it appears in (rank is 1-based).
    Returns a list of hit dicts, best-first, with per-retriever provenance.
    """
    cos = cosine_search(query, pool)
    lex = bm25_search(query, pool)

    cos_rank = {cid: r for r, (cid, _) in enumerate(cos, start=1)}
    lex_rank = {cid: r for r, (cid, _) in enumerate(lex, start=1)}
    cos_score = dict(cos)
    lex_score = dict(lex)

    fused = {}
    for cid in set(cos_rank) | set(lex_rank):
        s = 0.0
        if cid in cos_rank:
            s += w_semantic * 1.0 / (k_rrf + cos_rank[cid])
        if cid in lex_rank:
            s += w_lexical * 1.0 / (k_rrf + lex_rank[cid])
        fused[cid] = s

    ranked = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:k]
    hits = []
    for cid, fscore in ranked:
        c = chunk_by_id[cid]
        hits.append({
            "chunk_id": cid,
            "fused_score": fscore,
            "cosine_rank": cos_rank.get(cid),
            "bm25_rank": lex_rank.get(cid),
            "cosine_sim": cos_score.get(cid),
            "bm25_score": lex_score.get(cid),
            "document_name": c["document_name"],
            "section_title": c["section_title"],
            "page_number": c["page_number"],
            "source_url": c["source_url"],
            "text": c["text"],
        })
    return hits


# Public entry point Day-3 generation will import
def hybrid_search(query: str, k: int = 5):
    """Default hybrid retriever: 0.7 semantic / 0.3 lexical, weighted RRF."""
    return weighted_rrf(query, k=k)


print("Retrievers ready: cosine_search / bm25_search / weighted_rrf / hybrid_search")

Retrievers ready: cosine_search / bm25_search / weighted_rrf / hybrid_search


## Step 4 — Side-by-side: cosine vs BM25 vs hybrid

The same 8 clinical questions, each run through all three retrievers so you can
*see* what fusion changes. Watch the `out_of_scope` and threshold queries especially.

In [5]:
TEST_QUESTIONS = [
    ("direct",       "What is the diagnostic threshold for albuminuria in CKD?"),
    ("direct",       "When should an SGLT2 inhibitor be started in a patient with CKD and type 2 diabetes?"),
    ("direct",       "What are the GFR categories G1 through G5?"),
    ("multi",        "What blood pressure target is recommended for adults with CKD and albuminuria?"),
    ("multi",        "Which drug class is first-line for CKD with hypertension and proteinuria?"),
    ("edge",         "How should potassium be monitored when starting a mineralocorticoid receptor antagonist?"),
    ("edge",         "How often should eGFR be checked in a CKD patient?"),
    ("out_of_scope", "What is the recommended treatment for acute appendicitis?"),
]


def short(cid_hit, doc_field="document_name"):
    return cid_hit


def label(cid):
    c = chunk_by_id[cid]
    return f"{cid:16} p{c['page_number']:<3} {c['section_title'][:42]}"


for category, q in TEST_QUESTIONS:
    print("=" * 100)
    print(f"[{category.upper()}]  {q}")
    print("=" * 100)

    cos = cosine_search(q, 3)
    lex = bm25_search(q, 3)
    hyb = weighted_rrf(q, 3)

    print("  COSINE (semantic):")
    for cid, sc in cos:
        print(f"     sim={sc:.3f}  {label(cid)}")
    print("  BM25 (lexical):")
    for cid, sc in lex:
        print(f"     bm25={sc:5.2f}  {label(cid)}")
    print("  HYBRID (0.7/0.3 RRF):")
    for h in hyb:
        prov = f"cos#{h['cosine_rank'] or '-'} bm25#{h['bm25_rank'] or '-'}"
        print(f"     fused={h['fused_score']:.5f} [{prov:14}] {label(h['chunk_id'])}")
    print()

[DIRECT]  What is the diagnostic threshold for albuminuria in CKD?


  COSINE (semantic):
     sim=0.760  kdigo_p155_c16   p155 Chapter 6: Research recommendations
     sim=0.759  kdigo_p155_c17   p155 Chapter 6: Research recommendations
     sim=0.755  kdigo_p83_c12    p83  2.2 Risk prediction in people with CKD
  BM25 (lexical):
     bm25=13.68  nice_p51_c06     p51  Terms used in this guideline
     bm25=12.04  kdigo_p155_c15   p155 Chapter 6: Research recommendations
     bm25=12.01  kdigo_p155_c16   p155 Chapter 6: Research recommendations
  HYBRID (0.7/0.3 RRF):
     fused=0.01624 [cos#1 bm25#3  ] kdigo_p155_c16   p155 Chapter 6: Research recommendations
     fused=0.01525 [cos#5 bm25#7  ] nice_p29_c01     p29  1.7 Diagnosing and assessing anaemia
     fused=0.01515 [cos#6 bm25#6  ] nice_p51_c19     p51  Terms used in this guideline

[DIRECT]  When should an SGLT2 inhibitor be started in a patient with CKD and type 2 diabetes?


  COSINE (semantic):
     sim=0.822  kdigo_dm_p38_c34 p38  1.3 Sodium–glucose cotransporter-2 inhibit
     sim=0.814  kdigo_dm_p78_c01 p78  4.1 Metformin
     sim=0.811  kdigo_dm_p38_c35 p38  1.3 Sodium–glucose cotransporter-2 inhibit
  BM25 (lexical):
     bm25=18.55  nice_p23_c03     p23  1.6 Pharmacotherapy
     bm25=17.07  kdigo_p99_c12    p99  3.7 Sodium-glucose cotransporter-2 inhibit
     bm25=16.71  kdigo_dm_p76_c02 p76  Chapter 4: Glucose-lowering therapies in p
  HYBRID (0.7/0.3 RRF):
     fused=0.01443 [cos#9 bm25#10 ] kdigo_dm_p38_c37 p38  1.3 Sodium–glucose cotransporter-2 inhibit
     fused=0.01422 [cos#6 bm25#23 ] kdigo_p99_c01    p99  3.7 Sodium-glucose cotransporter-2 inhibit
     fused=0.01397 [cos#16 bm25#3 ] kdigo_dm_p76_c02 p76  Chapter 4: Glucose-lowering therapies in p

[DIRECT]  What are the GFR categories G1 through G5?


  COSINE (semantic):
     sim=0.769  nice_p51_c02     p51  Terms used in this guideline
     sim=0.751  nice_p13_c02     p13  1.2 Classification of CKD in adults
     sim=0.743  nice_p15_c02     p15  1.3 Frequency of monitoring
  BM25 (lexical):
     bm25=14.07  kdigo_p15_c19    p15  foreword
     bm25=12.40  kdigo_p119_c01   p119 3.15.1 Lipid management
     bm25=11.27  kdigo_p119_c03   p119 3.15.1 Lipid management
  HYBRID (0.7/0.3 RRF):
     fused=0.01564 [cos#1 bm25#12 ] nice_p51_c02     p51  Terms used in this guideline
     fused=0.01514 [cos#2 bm25#18 ] nice_p13_c02     p13  1.2 Classification of CKD in adults
     fused=0.01494 [cos#4 bm25#15 ] kdigo_p15_c20    p15  foreword

[MULTI]  What blood pressure target is recommended for adults with CKD and albuminuria?


  COSINE (semantic):
     sim=0.847  nice_p23_c02     p23  1.6 Pharmacotherapy
     sim=0.817  kdigo_p97_c02    p97  3.4 Blood pressure control
     sim=0.817  kdigo_p97_c01    p97  3.4 Blood pressure control
  BM25 (lexical):
     bm25=15.72  nice_p23_c02     p23  1.6 Pharmacotherapy
     bm25=15.61  kdigo_p97_c01    p97  3.4 Blood pressure control
     bm25=15.11  nice_p51_c16     p51  Terms used in this guideline
  HYBRID (0.7/0.3 RRF):
     fused=0.01639 [cos#1 bm25#1  ] nice_p23_c02     p23  1.6 Pharmacotherapy
     fused=0.01595 [cos#3 bm25#2  ] kdigo_p97_c01    p97  3.4 Blood pressure control
     fused=0.01570 [cos#4 bm25#3  ] nice_p51_c16     p51  Terms used in this guideline

[MULTI]  Which drug class is first-line for CKD with hypertension and proteinuria?


  COSINE (semantic):
     sim=0.730  kdigo_p91_c02    p91  3.2.1 Avoiding use of tobacco products
     sim=0.724  nice_p23_c04     p23  1.6 Pharmacotherapy
     sim=0.718  kdigo_dm_p30_c08 p30  1.1 Comprehensive diabetes and CKD managem
  BM25 (lexical):
     bm25=15.93  kdigo_dm_p30_c05 p30  1.1 Comprehensive diabetes and CKD managem
     bm25=14.95  kdigo_dm_p49_c11 p49  1.4 Mineralocorticoid receptor antagonists
     bm25=14.73  kdigo_dm_p49_c14 p49  1.4 Mineralocorticoid receptor antagonists
  HYBRID (0.7/0.3 RRF):
     fused=0.01609 [cos#1 bm25#5  ] kdigo_p91_c02    p91  3.2.1 Avoiding use of tobacco products
     fused=0.01580 [cos#3 bm25#4  ] kdigo_dm_p30_c08 p30  1.1 Comprehensive diabetes and CKD managem
     fused=0.01546 [cos#2 bm25#12 ] nice_p23_c04     p23  1.6 Pharmacotherapy

[EDGE]  How should potassium be monitored when starting a mineralocorticoid receptor antagonist?


  COSINE (semantic):
     sim=0.789  kdigo_dm_p33_c10 p33  1.2 Renin-angiotensin system (RAS) blockad
     sim=0.774  kdigo_dm_p49_c16 p49  1.4 Mineralocorticoid receptor antagonists
     sim=0.765  nice_p23_c05     p23  1.6 Pharmacotherapy
  BM25 (lexical):
     bm25=17.80  kdigo_p104_c06   p104 3.8 Mineralocorticoid receptor antagonists
     bm25=17.19  kdigo_dm_p49_c03 p49  1.4 Mineralocorticoid receptor antagonists
     bm25=17.06  kdigo_p111_c03   p111 3.11.5 Dietary considerations
  HYBRID (0.7/0.3 RRF):
     fused=0.01609 [cos#1 bm25#5  ] kdigo_dm_p33_c10 p33  1.2 Renin-angiotensin system (RAS) blockad
     fused=0.01586 [cos#4 bm25#1  ] kdigo_p104_c06   p104 3.8 Mineralocorticoid receptor antagonists
     fused=0.01540 [cos#3 bm25#10 ] nice_p23_c05     p23  1.6 Pharmacotherapy

[EDGE]  How often should eGFR be checked in a CKD patient?


  COSINE (semantic):
     sim=0.794  kdigo_p81_c02    p81  2.1 Overview on monitoring for progression
     sim=0.783  nice_p51_c13     p51  Terms used in this guideline
     sim=0.775  kdigo_p99_c01    p99  3.7 Sodium-glucose cotransporter-2 inhibit
  BM25 (lexical):
     bm25=11.68  kdigo_p98_c02    p98  3.6 Renin-angiotensin system inhibitors
     bm25=10.20  nice_p29_c01     p29  1.7 Diagnosing and assessing anaemia
     bm25=10.04  kdigo_dm_p30_c07 p30  1.1 Comprehensive diabetes and CKD managem
  HYBRID (0.7/0.3 RRF):
     fused=0.01540 [cos#2 bm25#13 ] nice_p51_c13     p51  Terms used in this guideline
     fused=0.01498 [cos#8 bm25#4  ] nice_p51_c15     p51  Terms used in this guideline
     fused=0.01473 [cos#4 bm25#19 ] nice_p15_c01     p15  1.3 Frequency of monitoring

[OUT_OF_SCOPE]  What is the recommended treatment for acute appendicitis?


  COSINE (semantic):
     sim=0.626  kdigo_p115_c07   p115 3.14 Hyperuricemia
     sim=0.611  kdigo_dm_p94_c12 p94  5.2 Team-based integrated care
     sim=0.609  kdigo_dm_p94_c15 p94  5.2 Team-based integrated care
  BM25 (lexical):
     bm25= 9.89  nice_p51_c08     p51  Terms used in this guideline
     bm25= 8.27  nice_p18_c01     p18  1.4 Information and education for people w
     bm25= 7.46  nice_p18_c02     p18  1.4 Information and education for people w
  HYBRID (0.7/0.3 RRF):
     fused=0.01558 [cos#1 bm25#13 ] kdigo_p115_c07   p115 3.14 Hyperuricemia
     fused=0.01362 [cos#12 bm25#17] nice_p51_c09     p51  Terms used in this guideline
     fused=0.01275 [cos#17 bm25#22] kdigo_p123_c03   p123 3.15.3 Invasive versus intensive medical t



## Step 5 — Alpha sweep on one query

See how the semantic/lexical weight changes the top-3. Use this later (with the eval set)
to pick the winning weight from Precision@K instead of intuition.

In [6]:
SWEEP_QUERY = "What is the diagnostic threshold for albuminuria in CKD?"

print(f'QUERY: "{SWEEP_QUERY}"\n')
for w_sem in [1.0, 0.7, 0.5, 0.3, 0.0]:
    w_lex = round(1.0 - w_sem, 1)
    hits = weighted_rrf(SWEEP_QUERY, k=3, w_semantic=w_sem, w_lexical=w_lex)
    tag = "semantic-only" if w_sem == 1.0 else "lexical-only" if w_sem == 0.0 else ""
    print(f"w_semantic={w_sem:.1f} / w_lexical={w_lex:.1f}  {tag}")
    for h in hits:
        print(f"    {label(h['chunk_id'])}")
    print()

QUERY: "What is the diagnostic threshold for albuminuria in CKD?"

w_semantic=1.0 / w_lexical=0.0  semantic-only
    kdigo_p155_c16   p155 Chapter 6: Research recommendations
    kdigo_p155_c17   p155 Chapter 6: Research recommendations
    kdigo_p83_c12    p83  2.2 Risk prediction in people with CKD



w_semantic=0.7 / w_lexical=0.3  
    kdigo_p155_c16   p155 Chapter 6: Research recommendations
    nice_p29_c01     p29  1.7 Diagnosing and assessing anaemia
    nice_p51_c19     p51  Terms used in this guideline

w_semantic=0.5 / w_lexical=0.5  
    kdigo_p155_c16   p155 Chapter 6: Research recommendations
    nice_p29_c01     p29  1.7 Diagnosing and assessing anaemia
    nice_p51_c19     p51  Terms used in this guideline



w_semantic=0.3 / w_lexical=0.7  
    kdigo_p155_c16   p155 Chapter 6: Research recommendations
    nice_p51_c19     p51  Terms used in this guideline
    nice_p29_c01     p29  1.7 Diagnosing and assessing anaemia

w_semantic=0.0 / w_lexical=1.0  lexical-only
    nice_p51_c06     p51  Terms used in this guideline
    kdigo_p155_c15   p155 Chapter 6: Research recommendations
    kdigo_p155_c16   p155 Chapter 6: Research recommendations



## Step 6 — Try your own query

Change `MY_QUERY` and re-run. Shows the hybrid top-K with full provenance and citation.

In [7]:
MY_QUERY = "Should asymptomatic adults be screened for chronic kidney disease?"
TOP_K = 5

print(f'QUERY: "{MY_QUERY}"\n')
for rank, h in enumerate(hybrid_search(MY_QUERY, k=TOP_K), 1):
    prov = f"cosine#{h['cosine_rank'] or '-'}, bm25#{h['bm25_rank'] or '-'}"
    print(f"[{rank}] fused={h['fused_score']:.5f}   (from {prov})")
    print(f"    {h['document_name']}")
    print(f"    section : {h['section_title']}  |  page {h['page_number']}")
    print(f"    chunk_id: {h['chunk_id']}")
    print("    " + h["text"].strip()[:280].replace("\n", " ") + "...")
    print("-" * 100)

QUERY: "Should asymptomatic adults be screened for chronic kidney disease?"



[1] fused=0.01609   (from cosine#1, bm25#5)
    USPSTF Screening for Chronic Kidney Disease
    section : Balance of Harms and Benefits  |  page 2
    chunk_id: uspstf_p2_c02
    Screening for chronic kidney disease: clinical summary of U.S. Preventive Services Task Force Recommendation. SCREENING FOR CHRONIC KIDNEY DISEASE CLINICAL SUMMARY OF U.S.  PREVENTIVE SERVICES TASK FORCE RECOMMENDATION Population Recommendation Risk Assessment Screening Tests Oth...
----------------------------------------------------------------------------------------------------
[2] fused=0.01598   (from cosine#2, bm25#4)
    USPSTF Screening for Chronic Kidney Disease
    section : Benefits of Detection and Early Intervention  |  page 1
    chunk_id: uspstf_p1_c02
    Testing for and monitoring CKD for the purpose of chronic disease management (including testing and monitoring patients with diabetes or hypertension) are not covered by this recommendation.  See the Clinical Considerations section for sugges

## Summary

Hybrid retrieval is live: **weighted RRF, 0.7 semantic / 0.3 lexical**, fusing MedEmbed cosine
with BM25 over the 742-chunk index. `hybrid_search(query, k)` is the entry point for Day-3
grounded generation.

**Nothing upstream was modified** — parser, chunker, and embedder are untouched. This notebook
only reads `all_chunks.jsonl` and the Chroma index.

**Day-2 next:** build a 15–20 question labeled eval set (from the KDIGO Summary of Recs), then
sweep the weight and Top-K against Precision@3 / Precision@5 / MRR and lock the winning config.